# SMA Zoomcamp 2026 — Module 5 Homework (New Research Ideas)

Three distinct stock-research ideas, one per question. Each follows the required
six-part structure: **Market & scope · Investment thesis · Potential edge ·
Data & sources · Research approach · Validation & risks.** The three ideas are
deliberately different in mechanism: a local-language / low-coverage idea, a
supply-chain lead-lag pairs idea, and an alternative-data demand signal.


---
## Question 1 — Research Idea #1: German MDAX mid-caps & German-language disclosure lag

### Market and scope
**Universe:** the ~50 stocks in the German MDAX index (mid-cap industrial,
software, medtech, and consumer companies listed on XETRA, e.g. Zalando,
HelloFresh, Nemetschek, Kion, Delivery Hero, Carl Zeiss Meditec). **Horizon:**
~30–60 trading days, designed to capture the gap between a German-language
corporate event (ad-hoc disclosure, BaFin voting-rights notice, trade-fair
product announcement, or preliminary quarterly "pre-release") and the moment
English-language news wires and sell-side notes catch up.

### Investment thesis
German mid-caps have materially thinner English-language analyst coverage than
DAX blue chips (median ~6 analysts vs ~25 for DAX), yet many of the most
value-relevant disclosures appear first in German — ad-hoc announcements under
the German Securities Trading Act (WpHG), preliminary results ("vorläufige
Zahlen"), Hannover Messe / IAA / MEDICA product launches, and regional
newspaper reporting on factory orders. I expect that a systematically detected
positive (negative) German-language disclosure that is **not** picked up by
Reuters/Dow Jones English headlines within 24 hours is followed by a positive
(negative) drift of 2–5% over the following 4–8 weeks, as international
investors and sell-side analysts update their models.

### Potential edge
(1) Language barrier — most global quant pipelines ingest English-only news, so
German-only signals are underweighted. (2) Low coverage means slower analyst
reaction: MDAX consensus updates lag DAX by a median of ~10 days after
preliminary releases. (3) Rule-based ad-hoc filings are published instantly on
the company's investor-relations page and BaFin's portal, hours before an
English-language summary appears (if one ever does). The edge is informational
speed on a small-cap-ish segment that is too small for large quant funds to
bother with but liquid enough for retail/small-fund execution.

### Data and sources
* **BaFin / Unternehmensregister** ad-hoc disclosures (free, verified, timestamps).
* **DGAP / EQS News** feeds — same disclosures, machine-readable API.
* **XETRA daily prices / volumes** from the existing parquet dataset extended with MDAX
  tickers via Yahoo Finance / Stereois or the Deutsche Börse public dataset.
* **English-news cross-check** — Reuters/Dow Jones/Bloomberg headlines via
  EventRegistry or GDELT to tag whether the disclosure was covered in English.
* Sell-side **revision dates** from Refinitiv IBES (need to check access — fallback:
  count changes in consensus EPS estimates from publicly available analyst
  reports/Onvista).
*Sources still to verify:* EventRegistry German-vs-English tagging accuracy for
corporate events; BaFin API rate limits; XETRA historical bid-ask spreads for
MDAX names.

### Research approach
Simple event study — not an ML model to start. For each MDAX issuer, collect
every ad-hoc / "vorläufige Zahlen" disclosure over 2018–2025. For each event:

1. Tag whether an English-language article on the same topic exists within
   24 hours (binary `english_coverage` flag using keyword overlap + date window).
2. Compute abnormal return vs. CDAX over [t+1, t+40] trading days.
3. The signal = "German-only positive/negative disclosure", classified by a
   small fine-tuned German sentiment classifier (or, simpler, a rule-based
   keyword list around guidance raised/lowered, beats/misses).
4. The trade = go long the positive German-only events, short the negative
   ones (or use long-only vs. a MDAX benchmark if shorting mid-caps is costly),
   sized by inverse volatility.

The simplicity is intentional — if the drift exists without fancy feature
engineering, the edge is real; if it requires an LLM with 200 features it's
more likely overfitting.

### Validation and risks
* **Benchmark:** buy-and-hold MDAX, and a "placebo" portfolio of German
  disclosures that *did* get English coverage within 24 hours. If both portfolios
  produce similar drift, the effect is just post-earnings drift, not a language
  edge.
* **Reject the idea if** BHAR[t+1, t+40] is < 1.5% net of spread + 0.2% fees per
  round trip, or if the effect disappears after 2022 (MiFID II unbundling of
  research may have already closed the gap).
* **Key risks:** liquidity — some MDAX names have < €20m/day turnover, so
  slippage could eat the drift; ad-hoc disclosures cluster around earnings so
  signal correlation is high; German disclosure rules changed in Jan 2025 with
  ESEF adoption, which may change the filing taxonomy.


---
## Question 2 — Research Idea #2: Lead-lag pairs trade in EU semiconductor equipment (ASML / ASMI / BESI)

### Market and scope
**Universe:** three European semiconductor-equipment names — **ASML** (NL,
EUV lithography, large-cap), **ASM International (ASMI)** (NL, front-end
deposition/epitaxy, mid-cap), **BE Semiconductors (BESI)** (NL, back-end die
attach/plating, small/mid-cap). They serve the same customer set (TSMC, Samsung,
Intel, foundries) and the same memory/logic capex cycle, but market cap differs
by ~8× between ASML and BESI. **Horizon:** 2–15 trading days, i.e. capturing
earnings-guidance transmission rather than a long-term fundamentals bet.

### Investment thesis
All three names live off the same WFE (wafer fab equipment) capex cycle, but
they report earnings on different dates, and institutional attention is skewed
heavily toward ASML. When ASML gives a strong (or weak) quarterly order-intake
guidance, the information is economically relevant for ASMI and BESI — their
backlogs, lead times and pricing power are driven by the same fab spending
decisions — yet ASMI/BESI prices don't fully price the news for several trading
days, particularly when ASML reports early in the earnings calendar. I expect
that a long/short pair (long BESI/ASMI short ASML, or vice versa, conditioned
on the sign of ASML's guidance surprise) generates positive 1–2 week returns
that are uncorrelated to market beta.

### Potential edge
* **Staggered reporting calendars** in EU semi equipment mean information
  about the same capex cycle is revealed at different times.
* **Sell-side coverage imbalance** — ~45 analysts cover ASML; ~20 cover ASMI;
  ~12 cover BESI. The "transmission" from ASML's guidance revision to an ASMI/BESI
  target-price change takes days, not minutes.
* **Index-fund flows** — ASML is ~15–20% of several EU tech ETFs; BESI isn't in
  the big ones. Large ASML post-earnings moves drive passive flows that distort
  relative pricing temporarily.
* **Mechanism is not fragile "statistical cointegration"** — it's a
  supply-chain news-lead-lag, which has an economic reason to exist rather than
  a purely statistical one.

### Data and sources
* **Earnings calendar dates** + reported guidance (orders/backlog, next-quarter
  revenue guide) for all three names from 2015–2025. Publicly available from
  company IR pages (verified).
* **Daily total returns, volumes, bid/ask** for ASML/ASMI/BESI (Euronext
  Amsterdam) — Yahoo Finance or Investing.com daily files.
* **WFE capex quarterly data** from SEMI (trade group, released with a lag —
  useful as the macro control, not a trade signal).
* **Supply-chain link confirmation:** check cross-revenue exposure via FactSet
  supply-chain or, simpler, count shared customers in each company's annual
  report customer lists (verified: TSMC/Samsung/Intel are top-3 for all three).
*Sources still to verify:* intraday prints for the three names to measure
gap-open vs. drift; options-implied move on earnings day as a better "surprise"
normaliser than just % price change.

### Research approach
1. Define `ASML_surprise` = ASML 2-day return around its earnings release
   (earnings day + next day), signed by whether management raised/lowered
   forward guidance (binary check on the press release).
2. In the window from ASML earnings to ASMI's/BESI's earnings (typically 1–3
   weeks later), go long the lagging name if ASML surprised positive, short if
   negative, hedged with an equal-Size short/long in ASML so the trade is
   market-neutral (pair "ASMI vs ASML" and "BESI vs ASML").
3. Exit either when the lagging name reports (the information is priced in) or
   after 15 trading days, whichever is sooner.
4. Baseline: a naive cointegration-based pair on the same three names without
  using the earnings signal — the standard pairs-trading placebo.

### Validation and risks
* **Benchmarks:** (a) buy-and-hold equal-weight basket; (b) naive rolling-zscore
  pairs trade on the same names with no earnings conditioning. The signal must
  beat both after costs.
* **Reject if** post-earnings drift in the lagging name is < 1.2% net of
  spreads and 0.2% fees — at that level the trade is just noise around the
  spread. Also reject if the effect flips in the post-2022 semiconductor downcycle.
* **Risks:** (i) ASML and BESI have partial business overlap but BESI's backend
  cycle sometimes decouples from lithography (packaging vs. lithography are
  different spend cycles); (ii) Euronext short-borrowing costs for BESI can
  spike; (iii) "everyone knows ASML leads" — need to check that the lagging
  drift still exists post-2023 (it may have been arbed away already; check with
  rolling out-of-sample windows).


---
## Question 3 — Research Idea #3: Steam concurrent players + Twitch viewership as a launch-revenue nowcast for US/EU game publishers

### Market and scope
**Universe:** publicly listed Western game publishers with heavy PC/Steam
exposure — Electronic Arts (EA), Take-Two (TTWO), Ubisoft (UBI.PA), CD Projekt
(CDR.WA), THQ Nordic/Embracer (EMBRAC-B.ST), Frontier (FDEV.L), Paradox
(PDX.ST). Focus on *major launches* and DLC/expansion releases for
games-as-a-service titles (e.g. GTA Online updates, FIFA/EA FC cycles, The
Sims DLCs, Assassin's Creed, Cyberpunk expansions). **Horizon:** 30–45 trading
days after a major launch / expansion release.

### Investment thesis
For game publishers, the first 2–4 weeks of post-launch player engagement is a
direct, measurable proxy for attach rate, DLC attach, MTX (microtransaction)
revenue, and ultimately the quarter's revenue beat/miss. Management guides
conservatively at launch; sell-side analysts revise slowly because they wait
for monthly NPD / Sensor Tower / SuperData reports (4–8 week lag). But real-time
public data — **Steam concurrent player counts** (hourly) and **Twitch
viewership** (minute-level) — is available within hours of launch and is a
strong leading indicator of engagement. I expect that games whose post-launch
Steam concurrency + Twitch viewership beats the pre-launch beta (or previous
title in a franchise) by > 20% are followed by publisher outperformance of 4–8%
over the 6 weeks following launch, as consensus is revised upward.

### Potential edge
* **Data is public but not in standard quant data feeds** — SteamDB/SteamCharts
  and TwitchTracker expose the data free but aren't on Bloomberg/Refinitiv in
  structured form, so traditional fundamental analysts don't systematically
  ingest them.
* **Lead over sell-side:** analyst revisions tend to come after the first monthly
  NPD or quarterly earnings, while player data is real-time.
* **Beyond the "growing industry" story:** the signal is *relative* engagement
  vs. the pre-launch baseline / prior title, not "people play games, gaming is
  growing". That separates product-level success from sector beta.
* Past precedent is observable: CD Projekt around Cyberpunk's 2077 post-launch
  recovery and the Edgerunners update had real-time engagement signals leading
  the stock move by multiple weeks.

### Data and sources
* **Steam Charts / SteamDB API** — concurrent players for each title, hourly,
  historically back to ~2012 (verified; publicly scrapable, CC-BY).
* **Twitch viewership** via TwitchTracker or SullyGnome (per-title viewers,
  hours watched; free, historical back to ~2015).
* **Publisher daily prices** (Nasdaq/Euronext/Warsaw) from Yahoo Finance; volumes
  + bid/ask.
* **Launch calendar & DLC dates** from Wikipedia/SteamDB/MobyGames (manually
  verifiable for ~100 major launches over 2015–2025).
* **Reported revenue / earnings beats** from 10-K/10-Q and earnings press
  releases (ground truth).
*Sources still to verify:* SteamDB historical data licensing for commercial
research (fine for academic/personal); whether console-only titles (e.g. Sony,
Nintendo) can be included (they can't — this signal is Steam-only, which is a
known limitation).

### Research approach
Event study per launch. For each major launch / expansion:

1. Build a `launch_score` = (average concurrent players in days 3–14 post-launch
   / average concurrent players in the open-beta or early-access week) combined
   with a Twitch component (average hours watched in days 3–14 / pre-launch
   beta weeks). Use a z-score normalised within the franchise.
2. Label a launch as "beat" if `launch_score > 1.2` (20% engagement upside vs.
   baseline), "miss" if `< 0.8`.
3. For beat events, go long the publisher at the close of day 14 (when the first
  robust engagement reading is in) and hold for 30 trading days; hedge with a
  short position in a gaming ETF (e.g. HERO) or the S&P 500 to remove sector
  beta. Miss events → long HERO / short publisher.
4. Compare to (a) a "buy every launch" portfolio, (b) a pre-order/sales-chart
  signal from Steam top-sellers (which is already widely followed).

I'd start with a simple threshold rule rather than a model so the relationship
is interpretable; if the threshold rule works, a gradient-boosted model can
additionally weight Twitch vs Steam vs franchise history.

### Validation and risks
* **Benchmark:** long-only equal-weight portfolio of the same publishers
  (timing-only alpha); placebo using *only* Steam wishlist counts (already
  public and heavily watched — expect this to add nothing).
* **Reject if** long-short alpha is < 2% per event net of 0.2% fees, or if the
  signal is entirely driven by one outlier (e.g. a single CD Projekt event).
  Also reject if Steam top-sellers rank — the signal everyone already watches
  — has equal predictive power (that means no incremental edge from real-time
  concurrency).
* **Risks:** (i) many big launches are console-first (GTA, CoD) where Steam data
  misses the bulk of revenue; restrict to Steam-relevant titles. (ii)
  Microtransaction-driven games (EA FC, Fortnite) may have stable concurrency
  but variable spending — complement with Steam review sentiment as a
  retention proxy. (iii) Publisher stock moves are often dominated by the
  *guidance* given on earnings day, not the launch itself — size positions
  small to avoid binary event risk. (iv) Game launches are on known dates, so
  there is announcement-day run-up; only enter *after* day 3 when real
  engagement data exists.


---
## Why these three are different

| Idea           | Core mechanism             | Data type              | Horizon     |
|----------------|----------------------------|------------------------|-------------|
| #1 German MDAX | Language / coverage lag    | German ad-hoc filings  | 30–60 days  |
| #2 EU semi pairs| Supply-chain earnings lead-lag | Earnings calendars, prices | 2–15 days |
| #3 Game engagement | Alt-data nowcast        | Steam + Twitch time series | 30–45 days |

Each uses a different data source, a different economic mechanism, and a
different holding period — not just three variants of "fit an RF on RSI."
